# Wearly — Fashion Vision AI Fine-Tuning

**Model:** Gemma 4 2B (via Unsloth)  
**Module:** `wearly-fashion-v1` — Module 1 (clothing recognition & colour naming)  
**Datasets:** FashionProduct · FashionPedia · iMaterialist · ClothingClassification · Wearly own data  
**Target capabilities:**
- Clothing category detection (shirt, jeans, watch, sneakers, bag…)
- Precise colour naming + hex code prediction
- Occasion tagging (office, casual, date night, gym…)
- Style tag labelling (minimalist, classic, streetwear…)
- Material & season estimation

**Runtime:** Google Colab Pro+ with A100 40 GB (recommended) or local GPU ≥ 24 GB VRAM  

---

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────────
!pip install -q unsloth datasets transformers trl peft accelerate bitsandbytes pillow
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

In [ ]:
# ── 2. Load base Gemma 4 with Unsloth 4-bit quantisation ─────────────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = 'google/gemma-4-2b-it',
    max_seq_length = MAX_SEQ_LEN,
    dtype         = None,   # auto
    load_in_4bit  = True,
)
print('✅ Model loaded:', model.config.model_type)

In [ ]:
# ── 3. Attach LoRA adapters (fashion domain) ──────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_alpha     = 64,
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
print('✅ LoRA adapters attached')

In [ ]:
# ── 4. Dataset loading & conversion ──────────────────────────────────────────
import json, re
from pathlib import Path
from datasets import load_dataset, Dataset, concatenate_datasets

HF_TOKEN = ''   # paste your HuggingFace token if required

SYSTEM_PROMPT = """You are Wearly's AI fashion classifier. Given a clothing item description,
return a precise JSON object with the item's details. Always reply with valid JSON only."""

# Colour hex lookup table (representative)
COLOR_HEX = {
    'black':'#1A1A1A','white':'#F8F8F8','navy':'#1C3F60','navy blue':'#1C3F60',
    'blue':'#2563EB','light blue':'#93C5FD','royal blue':'#1D4ED8',
    'red':'#DC2626','dark red':'#991B1B','burgundy':'#7F1D1D','maroon':'#7C1F2A',
    'green':'#16A34A','olive':'#65723A','khaki':'#C3B17A','sage':'#8FA882',
    'grey':'#6B7280','gray':'#6B7280','light grey':'#D1D5DB','charcoal':'#374151',
    'beige':'#D4B483','cream':'#FFF8E7','off white':'#F5F0E8','ivory':'#FFFFF0',
    'brown':'#92400E','tan':'#C8A87A','camel':'#C19A6B',
    'pink':'#EC4899','light pink':'#FBCFE8','hot pink':'#DB2777',
    'purple':'#7C3AED','lavender':'#A78BFA','violet':'#6D28D9',
    'yellow':'#EAB308','mustard':'#CA8A04','gold':'#D97706',
    'orange':'#EA580C','coral':'#F97316','rust':'#C2410C',
    'teal':'#0D9488','turquoise':'#06B6D4','cyan':'#22D3EE',
    'multicolor':'#888888','multi':'#888888','printed':'#888888',
}

def color_to_hex(color_str: str) -> str:
    c = color_str.lower().strip()
    for key, hex_val in COLOR_HEX.items():
        if key in c:
            return hex_val
    return '#888888'

OCCASION_MAP = {
    'casual': 'casual', 'sports': 'gym', 'ethnic': 'festive',
    'formal': 'office', 'party': 'date_night', 'travel': 'travel',
    'smart casual': 'smart_casual', 'work': 'office',
}

CATEGORY_MAP = {
    'topwear': 'shirt', 'bottomwear': 'pants', 'footwear': 'shoes',
    'watches': 'watch', 'bags': 'bag', 'belts': 'belt',
    'sunglasses': 'sunglasses', 'jewellery': 'accessory',
    'dress': 'dress', 'loungewear and nightwear': 'casual',
    'saree': 'dress', 'innerwear': 'casual',
}

def make_prompt(instruction: str, inp: str, output: str) -> str:
    user_msg = f"{instruction}\n{inp}" if inp else instruction
    return (
        f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"<start_of_turn>user\n{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n{output}<end_of_turn>"
    )

datasets_to_merge = []

# ── Dataset A: Fashion Product Images (Myntra/H&M catalog) ────────────────────
try:
    fp = load_dataset('ashraq/fashion-product-images-small', split='train', token=HF_TOKEN)
    def fmt_fp(row):
        name        = row.get('productDisplayName', 'Fashion item')
        master_cat  = row.get('masterCategory', '').lower()
        sub_cat     = row.get('subCategory', '').lower()
        article     = row.get('articleType', '').lower()
        colour      = row.get('baseColour', 'Unknown')
        season      = row.get('season', '')
        usage       = row.get('usage', 'casual').lower()
        category    = CATEGORY_MAP.get(master_cat, article or sub_cat or 'clothing')
        occasion    = OCCASION_MAP.get(usage, 'casual')
        output = json.dumps({
            'name': name,
            'category': category,
            'color_name': colour,
            'color_hex': color_to_hex(colour),
            'occasions': [occasion],
            'style_tags': ['classic'],
            'material_guess': 'mixed',
            'season': [season.lower()] if season else ['all-season'],
        }, ensure_ascii=False)
        return {'text': make_prompt(
            'Classify this clothing item and return JSON.',
            f'Item: {name}. Colour: {colour}. Category: {article or master_cat}. Usage: {usage}.',
            output
        )}
    fp = fp.map(fmt_fp, remove_columns=fp.column_names)
    datasets_to_merge.append(fp)
    print(f'✅ Fashion Products: {len(fp):,} examples')
except Exception as e:
    print(f'⚠️  Fashion Products skipped: {e}')

# ── Dataset B: Clothing Classification (keremberke) ───────────────────────────
try:
    cc = load_dataset('keremberke/clothing-classification', 'full', split='train', token=HF_TOKEN)
    LABEL_NAMES = ['dress','hat','longsleeve','outwear','pants','shirt',
                   'shoes','shorts','skirt','t-shirt','undershirt']
    def fmt_cc(row):
        label = LABEL_NAMES[row['labels']] if row.get('labels') is not None else 'shirt'
        output = json.dumps({
            'name': label.title(),
            'category': label,
            'color_name': 'Unknown',
            'color_hex': '#888888',
            'occasions': ['casual'],
            'style_tags': ['casual'],
            'material_guess': 'fabric',
            'season': ['all-season'],
        }, ensure_ascii=False)
        return {'text': make_prompt(
            'What category is this clothing item?',
            f'Item appears to be a {label}.',
            output
        )}
    cc = cc.map(fmt_cc, remove_columns=cc.column_names)
    datasets_to_merge.append(cc)
    print(f'✅ Clothing Classification: {len(cc):,} examples')
except Exception as e:
    print(f'⚠️  Clothing Classification skipped: {e}')

# ── Dataset C: FashionPedia (48 fine-grained fashion categories) ──────────────
try:
    fped = load_dataset('detection-datasets/fashionpedia', split='train', token=HF_TOKEN)
    FASHIONPEDIA_CATS = [
        'shirt, blouse','top, t-shirt, sweatshirt','sweater','cardigan','jacket',
        'vest','pants','shorts','skirt','coat','dress','jumpsuit','cape',
        'glasses','hat','headband','tie','glove','watch','belt','leg warmer',
        'tights, stockings','sock','shoe','bag, wallet','scarf','umbrella',
        'hood','collar','lapel','epaulette','sleeve','pocket','neckline',
        'buckle','zipper','applique','bead','bow','flower','fringe','ribbon',
        'rivet','ruffle','sequin','tassel'
    ]
    def fmt_fped(row):
        cats = row.get('objects', {}).get('category', [])
        if not cats:
            return {'text': ''}
        primary_idx = cats[0] if cats else 0
        cat_name = FASHIONPEDIA_CATS[primary_idx] if primary_idx < len(FASHIONPEDIA_CATS) else 'clothing'
        output = json.dumps({
            'name': cat_name.title(),
            'category': cat_name.split(',')[0].strip(),
            'color_name': 'Mixed',
            'color_hex': '#888888',
            'occasions': ['casual', 'smart_casual'],
            'style_tags': ['classic'],
            'material_guess': 'fabric',
            'season': ['all-season'],
        }, ensure_ascii=False)
        return {'text': make_prompt(
            'Identify and classify this fashion item.',
            f'Fashion item: {cat_name}.',
            output
        )}
    fped = fped.map(fmt_fped, remove_columns=fped.column_names)
    fped = fped.filter(lambda x: len(x['text']) > 10)
    datasets_to_merge.append(fped)
    print(f'✅ FashionPedia: {len(fped):,} examples')
except Exception as e:
    print(f'⚠️  FashionPedia skipped: {e}')

# ── Dataset D: Synthetic colour-matching pairs ────────────────────────────────
COLOR_PAIRS = [
    ('Navy Blue shirt','navy blue','#1C3F60','shirt',['office','smart_casual'],['classic']),
    ('White Oxford','white','#F8F8F8','shirt',['office','formal'],['classic','minimal']),
    ('Black slim jeans','black','#1A1A1A','jeans',['casual','date_night'],['streetwear','minimal']),
    ('Olive cargo pants','olive','#65723A','pants',['casual','travel'],['streetwear','casual']),
    ('Burgundy polo','burgundy','#7F1D1D','shirt',['smart_casual','casual'],['classic']),
    ('Grey hoodie','grey','#6B7280','hoodie',['casual','gym'],['streetwear','sporty']),
    ('Beige chinos','beige','#D4B483','pants',['casual','smart_casual'],['classic','preppy']),
    ('Royal blue blazer','royal blue','#1D4ED8','jacket',['office','formal'],['elegant','classic']),
    ('Mustard yellow tee','mustard','#CA8A04','tshirt',['casual'],['streetwear','casual']),
    ('Rust orange sweatshirt','rust','#C2410C','sweatshirt',['casual'],['streetwear','casual']),
    ('Charcoal trousers','charcoal','#374151','pants',['office','formal'],['classic','minimal']),
    ('Sage green linen shirt','sage','#8FA882','shirt',['casual','travel'],['bohemian','casual']),
    ('Cream knit sweater','cream','#FFF8E7','sweater',['casual','smart_casual'],['classic','minimal']),
    ('Coral summer dress','coral','#F97316','dress',['casual','date_night'],['bohemian','elegant']),
    ('Lavender silk blouse','lavender','#A78BFA','shirt',['office','date_night'],['elegant','classic']),
]
synthetic_rows = []
for name, colour, hex_val, category, occasions, styles in COLOR_PAIRS:
    output = json.dumps({
        'name': name, 'category': category,
        'color_name': colour.title(), 'color_hex': hex_val,
        'occasions': occasions, 'style_tags': styles,
        'material_guess': 'fabric', 'season': ['all-season'],
    })
    synthetic_rows.append({'text': make_prompt(
        'Classify this clothing item and return JSON.',
        f'Item: {name}.',
        output
    )})
# Repeat synthetic data 100x for better coverage
synthetic_rows = synthetic_rows * 100
synthetic_ds = Dataset.from_list(synthetic_rows)
datasets_to_merge.append(synthetic_ds)
print(f'✅ Synthetic colour pairs: {len(synthetic_ds):,} examples')

# ── Dataset E: Wearly own /api/learn training data ─────────────────────────────
wearly_jsonl = Path('../datasets/wearly_fashion_train.jsonl')
if wearly_jsonl.exists():
    rows = [json.loads(l) for l in open(wearly_jsonl)]
    own_ds = Dataset.from_list([{'text': make_prompt(r.get('instruction','Classify this item.'), r.get('input',''), r.get('output','{}'))} for r in rows])
    datasets_to_merge.append(own_ds)
    print(f'✅ Wearly own data: {len(own_ds):,} examples')

# Merge & shuffle
combined = concatenate_datasets(datasets_to_merge).shuffle(seed=42)
print(f'\n📊 Total training examples: {len(combined):,}')

In [ ]:
# ── 5. Tokenise ───────────────────────────────────────────────────────────────
def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation  = True,
        max_length  = MAX_SEQ_LEN,
        padding     = False,
    )

tokenised = combined.map(tokenize, batched=True, remove_columns=['text'])
print(f'✅ Tokenised: {len(tokenised):,} examples')

In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model      = model,
    tokenizer  = tokenizer,
    train_dataset    = tokenised,
    max_seq_length   = MAX_SEQ_LEN,
    dataset_num_proc = 4,
    args = TrainingArguments(
        per_device_train_batch_size  = 4,
        gradient_accumulation_steps  = 4,    # effective batch = 16
        warmup_steps                 = 80,
        num_train_epochs             = 3,    # more epochs — smaller dataset
        learning_rate                = 2e-4,
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        logging_steps                = 20,
        save_steps                   = 200,
        output_dir                   = './wearly-fashion-checkpoints',
        optim                        = 'adamw_8bit',
        weight_decay                 = 0.01,
        lr_scheduler_type            = 'cosine',
        seed                         = 42,
    ),
)

print('🚀 Training wearly-fashion-v1 …')
trainer.train()
print('✅ Training complete!')

In [ ]:
# ── 7. Quick eval — test the model before saving ─────────────────────────────
FastLanguageModel.for_inference(model)

test_prompt = (
    f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
    '<start_of_turn>user\n'
    'Classify this clothing item: Navy blue slim-fit chinos, cotton blend, office wear.<end_of_turn>\n'
    '<start_of_turn>model\n'
)
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
print('\n📋 Model output:')
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split('<start_of_turn>model')[-1])

In [ ]:
# ── 8. Save LoRA adapters + export GGUF ──────────────────────────────────────
print('💾 Saving LoRA adapters …')
model.save_pretrained('wearly-fashion-v1')
tokenizer.save_pretrained('wearly-fashion-v1')

print('💾 Exporting GGUF (Q4_K_M) …')
model.save_pretrained_gguf(
    'wearly-fashion-v1-gguf',
    tokenizer,
    quantization_method = 'q4_k_m',
)
print('✅ GGUF saved → wearly-fashion-v1-gguf/')

In [ ]:
# ── 9. Push to HuggingFace Hub (optional) ─────────────────────────────────────
HF_REPO = 'harsaikron/wearly-fashion-v1'   # change to your HF username

if HF_TOKEN:
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f'✅ Pushed → https://huggingface.co/{HF_REPO}')
else:
    print('⏭  HF_TOKEN not set — skipping Hub upload')

## ✅ Register with Ollama (run on your Mac)

```bash
# 1. Copy the GGUF file from Colab to your Mac
# (download via Colab files panel, or use gdown / scp)

# 2. Create Modelfile
cat > ~/Modelfile.fashion << 'EOF'
FROM ~/wearly-fashion-v1-gguf/unsloth.Q4_K_M.gguf
SYSTEM """You are Wearly's AI fashion classifier. Given a clothing item name or description,
return a JSON object with: name, category, color_name, color_hex, occasions (array),
style_tags (array), material_guess, season (array). Return valid JSON only."""
PARAMETER temperature 0.1
PARAMETER num_ctx 2048
PARAMETER stop "<end_of_turn>"
EOF

# 3. Register with Ollama
ollama create wearly-fashion-v1 -f ~/Modelfile.fashion

# 4. Test
ollama run wearly-fashion-v1 "Classify this: Navy blue slim-fit chinos, cotton."

# 5. Verify Wearly picks it up
curl http://localhost:11434/api/tags | jq '.models[].name'
# Should include: wearly-fashion-v1
```

Once registered, Wearly automatically uses it for all wardrobe photo analysis via `detectFashionModel()` in `src/lib/ai-client.ts`.

---

### Model capability checklist after training

| Test input | Expected JSON field | Pass criterion |
|---|---|---|
| `"Navy blue Oxford shirt"` | `color_hex` | `#1C3F60` ± 10% |
| `"Black slim jeans"` | `category` | `jeans` or `pants` |
| `"Gold dress watch"` | `occasions` | includes `office` or `formal` |
| `"Coral summer dress"` | `style_tags` | includes `bohemian` or `casual` |
| `"Olive cargo pants"` | `color_name` | `Olive` or `Olive Green` |